# 01 — Verify Pipeline

Smoke test: load catalog data, configure BacktestEngine, run NT’s built-in EMACross, verify fills.
If the final cell shows fills, the pipeline works end-to-end.

In [ ]:
# Cell 1: Imports + config
from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.examples.strategies.ema_cross import EMACross, EMACrossConfig
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine
from src.core import TS_INIT_DELTAS, bar_type_str
from utils import make_instrument_id, save_tearsheet

# ── Configuration ───────────────────────────────────────────────────
EXCHANGE         = "HYPERLIQUID"
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 100_000
SAVE_TEARSHEET   = False

# Standard
INSTRUMENT_ID = make_instrument_id(ASSET, EXCHANGE)
BAR_TYPE_STR  = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE         = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"Pipeline_{INSTRUMENT_ID}_{BAR_INTERVAL}"

catalog = ParquetDataCatalog(CATALOG_PATH)
print("Data types:", catalog.list_data_types())
print("Instruments:", [str(i.id) for i in catalog.instruments()])

In [ ]:
# Cell 2: Load bars + verify ts_init_delta
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])
print(f"Bar count: {len(bars)}")

first = bars[0]
last  = bars[-1]
print(f"First: {pd.Timestamp(first.ts_event, unit='ns', tz='UTC')}")
print(f"Last:  {pd.Timestamp(last.ts_event, unit='ns', tz='UTC')}")

# Verify ts_init_delta (critical for avoiding look-ahead bias)
print(f"\nFirst bar ts_event: {first.ts_event}")
print(f"First bar ts_init:  {first.ts_init}")
delta_ns = first.ts_init - first.ts_event
expected_delta = TS_INIT_DELTAS[BAR_INTERVAL]
print(f"Delta (ns): {delta_ns}")
assert delta_ns == expected_delta, (
    f"ts_init_delta wrong! Expected {expected_delta} ns, got {delta_ns}"
)
print(f"ts_init_delta OK ({BAR_INTERVAL})")

In [ ]:
# Cell 3: Configure engine
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print(f"Engine configured. Instrument: {instrument.id}")

In [ ]:
# Cell 4: Run with NT's built-in EMACross
config = EMACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_size=Decimal("0.01"),
    fast_ema_period=10,
    slow_ema_period=20,
)
strategy = EMACross(config=config)
engine.add_strategy(strategy)

engine.run()
print("Engine run complete.")

In [ ]:
# Cell 5: Print fills — if rows exist, pipeline is verified
fills = engine.trader.generate_order_fills_report()
print(f"Total fills: {len(fills)}")

if len(fills) > 0:
    print("\nPIPELINE VERIFIED \u2014 fills generated.")
    display(fills.head(10))
else:
    print("\nWARNING: No fills generated. Check strategy and data.")

In [ ]:
# Cell 6: Calculate statistics
account = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)

In [ ]:
# Cell 7: HTML Tearsheet
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"Pipeline Verify | EMACross | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# Cell 8: Cleanup
engine.dispose()
print("Engine disposed.")
